# 🔤 Tokenizer Lab - Thí Nghiệm Tokenization

## Mục tiêu bài học
- Hiểu cách Large Language Models (LLM) chia nhỏ văn bản thành tokens
- So sánh sự khác biệt giữa tokenization tiếng Việt và tiếng Anh
- Tính toán chi phí sử dụng API dựa trên số lượng tokens
- Tối ưu hóa prompt để tiết kiệm chi phí

## 📚 Phần 1: Giới thiệu về Tokenization

### Tokenization là gì?
Tokenization là quá trình chia nhỏ văn bản thành các đơn vị nhỏ hơn gọi là **tokens**. Đây là bước đầu tiên mà LLM thực hiện khi xử lý văn bản.

### Tại sao cần tokenization?
- LLM không xử lý trực tiếp văn bản, mà xử lý các con số (tokens)
- Mỗi token được chuyển đổi thành một vector số để model có thể hiểu
- Chi phí API được tính dựa trên số lượng tokens (input + output)

### Một số quy tắc cơ bản:
- 1 token ≈ 4 ký tự tiếng Anh
- 1 token ≈ ¾ từ tiếng Anh
- 1 từ tiếng Việt có thể tốn nhiều token hơn tiếng Anh (2-3 tokens)
- Khoảng trắng, dấu câu cũng tốn tokens

## 🛠️ Phần 2: Cài đặt và Chuẩn bị

Chúng ta sẽ sử dụng thư viện `tiktoken` - công cụ tokenization của OpenAI

In [3]:
# Cài đặt thư viện cần thiết
!pip install tiktoken pandas

  Using cached pandas-3.0.0-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.1-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.0-cp311-cp311-win_amd64.whl (9.9 MB)
Using cached numpy-2.4.1-cp311-cp311-win_amd64.whl (12.6 MB)
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Import thư viện
import tiktoken
import pandas as pd
from typing import List

## 🔬 Phần 3: Thử nghiệm Tokenization

### 3.1 Khởi tạo Tokenizer
Chúng ta sẽ sử dụng tokenizer của GPT-4 (encoding: `cl100k_base`)

In [4]:
# Khởi tạo tokenizer cho GPT-4/GPT-3.5-turbo
encoding = tiktoken.get_encoding("cl100k_base")

# Hoặc có thể khởi tạo theo tên model cụ thể
# encoding = tiktoken.encoding_for_model("gpt-4")

print("✅ Tokenizer đã được khởi tạo thành công!")

✅ Tokenizer đã được khởi tạo thành công!


### 3.2 Hàm tiện ích để phân tích tokens

In [5]:
def analyze_text(text: str, encoding) -> dict:
    """Phân tích văn bản và trả về thông tin về tokens"""
    tokens = encoding.encode(text)
    
    return {
        'text': text,
        'num_tokens': len(tokens),
        'num_characters': len(text),
        'chars_per_token': round(len(text) / len(tokens), 2) if len(tokens) > 0 else 0
    }

def display_tokens(text: str, encoding):
    """Hiển thị chi tiết tokens của văn bản"""
    result = analyze_text(text, encoding)
    
    print(f"📝 Văn bản: {result['text']}")
    print(f"   - Số ký tự: {result['num_characters']}")
    print(f"   - Số tokens: {result['num_tokens']}")
    print(f"   - TB: {result['chars_per_token']} ký tự/token")
    
    return result

### 3.3 Thử nghiệm với tiếng Anh

In [6]:
# Ví dụ tiếng Anh
english_text = "Hello, how are you today?"
print("TIẾNG ANH:")
result_en = display_tokens(english_text, encoding)

TIẾNG ANH:
📝 Văn bản: Hello, how are you today?
   - Số ký tự: 25
   - Số tokens: 7
   - TB: 3.57 ký tự/token


### 3.4 Thử nghiệm với tiếng Việt

In [7]:
# Ví dụ tiếng Việt
vietnamese_text = "Xin chào, bạn khỏe không?"
print("\nTIẾNG VIỆT:")
result_vi = display_tokens(vietnamese_text, encoding)


TIẾNG VIỆT:
📝 Văn bản: Xin chào, bạn khỏe không?
   - Số ký tự: 25
   - Số tokens: 12
   - TB: 2.08 ký tự/token


### 3.5 So sánh tiếng Anh vs tiếng Việt

In [8]:
# So sánh
print("\n" + "=" * 50)
print("SO SÁNH:")
print(f"Tiếng Anh: {result_en['num_tokens']} tokens")
print(f"Tiếng Việt: {result_vi['num_tokens']} tokens")
token_ratio = result_vi['num_tokens'] / result_en['num_tokens']
print(f"💡 Tiếng Việt tốn {token_ratio:.1f}x tokens!")


SO SÁNH:
Tiếng Anh: 7 tokens
Tiếng Việt: 12 tokens
💡 Tiếng Việt tốn 1.7x tokens!


## 💰 Phần 4: Tính toán chi phí API

### 4.1 Bảng giá các model phổ biến (tính đến 2026)

| Model | Input ($/1M tokens) | Output ($/1M tokens) |
|-------|---------------------|----------------------|
| GPT-4o | $2.50 | $10.00 |
| GPT-4o-mini | $0.15 | $0.60 |
| GPT-4 Turbo | $10.00 | $30.00 |
| Claude 3.5 Sonnet | $3.00 | $15.00 |
| Claude 3.5 Haiku | $0.80 | $4.00 |

In [11]:
# Bảng giá ($/1M tokens) - Cập nhật 2026
PRICING = {
    'gpt-4o': {'input': 2.5, 'output': 10.0},
    'gpt-4o-mini': {'input': 0.15, 'output': 0.60},
    'gpt-4-turbo': {'input': 10.0, 'output': 30.0},
    'claude-3.5-sonnet': {'input': 3.0, 'output': 15.0},
    'claude-3.5-haiku': {'input': 0.8, 'output': 4.0},
}

def calculate_cost(input_tokens: int, output_tokens: int, model: str = 'gpt-4o') -> dict:
    """Tính chi phí sử dụng API"""
    pricing = PRICING[model]
    input_cost = (input_tokens / 1_000_000) * pricing['input']
    output_cost = (output_tokens / 1_000_000) * pricing['output']
    
    return {
        'model': model,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_cost': input_cost + output_cost
    }

def display_cost(cost_info: dict):
    """Hiển thị chi phí"""
    print(f"💰 {cost_info['model']}: ${cost_info['total_cost']:.6f}")

### 4.2 Ví dụ tính chi phí cho một cuộc hội thoại

In [12]:
# Tính chi phí cho một câu hỏi
user_prompt = "Giải thích về AI"
ai_response = "AI là trí tuệ nhân tạo..."

input_tokens = len(encoding.encode(user_prompt))
output_tokens = len(encoding.encode(ai_response))

print("VÍ DỤ TÍNH CHI PHÍ:")
for model in ['gpt-4o-mini', 'gpt-4o', 'claude-3.5-sonnet']:
    cost = calculate_cost(input_tokens, output_tokens, model)
    display_cost(cost)

VÍ DỤ TÍNH CHI PHÍ:
💰 gpt-4o-mini: $0.000008
💰 gpt-4o: $0.000130
💰 claude-3.5-sonnet: $0.000189


## 🎯 BÀI TẬP

### Bài 1: Thử nghiệm văn bản của bạn

In [13]:
# Viết đoạn văn của bạn
my_text = "Đây là văn bản mẫu..."

my_result = display_tokens(my_text, encoding)

📝 Văn bản: Đây là văn bản mẫu...
   - Số ký tự: 21
   - Số tokens: 14
   - TB: 1.5 ký tự/token


### Bài 2: So sánh 2 prompt

In [14]:
# Viết 2 prompt khác nhau
prompt_1 = "Hãy giải thích chi tiết về..."
prompt_2 = "Giải thích về..."

r1 = analyze_text(prompt_1, encoding)
r2 = analyze_text(prompt_2, encoding)

print(f"Prompt 1: {r1['num_tokens']} tokens")
print(f"Prompt 2: {r2['num_tokens']} tokens")
print(f"Tiết kiệm: {r1['num_tokens'] - r2['num_tokens']} tokens")

Prompt 1: 14 tokens
Prompt 2: 8 tokens
Tiết kiệm: 6 tokens


### Bài 3: Tính chi phí dự án

In [15]:
# Tính chi phí cho chatbot
daily_users = 100
messages_per_user = 10
avg_input = 50
avg_output = 100

total_messages = daily_users * messages_per_user
cost_per_msg = calculate_cost(avg_input, avg_output, 'gpt-4o-mini')['total_cost']
daily_cost = cost_per_msg * total_messages

print(f"Chi phí/ngày: ${daily_cost:.2f}")
print(f"Chi phí/tháng: ${daily_cost * 30:.2f}")

Chi phí/ngày: $0.07
Chi phí/tháng: $2.03


## 📝 Tổng kết

**Những điều cần nhớ:**
1. Tiếng Việt tốn nhiều tokens hơn tiếng Anh (2-3x)
2. Chi phí = Input tokens + Output tokens
3. Cách tối ưu: Viết prompt ngắn gọn, chọn model phù hợp
4. Luôn tính toán chi phí trước khi triển khai

**Tài nguyên:**
- [OpenAI Tokenizer](https://platform.openai.com/tokenizer)
- [OpenAI Pricing](https://openai.com/pricing)